# Notebook 6 - Computer Vision

Roadmap role: Week 6. This Colab workflow preserves dataset provenance, official splits, model settings, raw outputs, and negative results. Agriculture-Vision is a semantic-segmentation benchmark; generic YOLO inference here is only the roadmap-required smoke-test baseline.

## 1. Clone The Week 6 Branch

Run this in a fresh Colab runtime. Licensed imagery and large checkpoints persist in the operator's private Google Drive; source code and nonrestricted reproducibility records remain in GitHub.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO = Path('/content/shepherd-ai')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'codex/week6-vision', 'https://github.com/cyberuniversal/shepherd-ai.git', str(REPO)], check=True)
os.chdir(REPO)
print('repository', Path.cwd())
print('commit', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## 2. Record The Runtime

Dataset acquisition, layout inspection, manifest construction, and label auditing may run in CPU debug mode. YOLO inference and future segmentation training remain T4-gated. A CPU debug run is not a model-performance experiment.

In [ ]:
import torch
CUDA_DEVICES = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
HAS_T4 = any('T4' in name for name in CUDA_DEVICES)
RUN_MODE = 't4_experiment' if HAS_T4 else 'cpu_debug'
print('run_mode', RUN_MODE)
print('cuda_devices', CUDA_DEVICES)
print('torch_version', torch.__version__)


## 3. Review And Accept Agriculture-Vision Terms

Official terms: https://intelinair-data-releases.s3.amazonaws.com/agriculture-vision/cvpr_paper_2020/Agriculture-Vision%20Dataset%20Terms%20of%20Use.pdf

Downloading signifies agreement. The terms permit limited non-commercial research use and prohibit redistribution. Only continue if you personally reviewed and accept them.

In [ ]:
TERMS_ACKNOWLEDGMENT = input('Type I ACCEPT AGRICULTURE-VISION TERMS after reviewing them: ').strip()
if TERMS_ACKNOWLEDGMENT != 'I ACCEPT AGRICULTURE-VISION TERMS':
    raise RuntimeError('Terms were not accepted; dataset acquisition stopped.')
print('Terms acknowledged for this Colab session.')


## 4. Mount The Private Google Drive Cache

The Agriculture-Vision terms prohibit redistribution, so the licensed dataset must not be committed to the public repository. This cell creates a private Drive cache that survives Colab runtime recycling and links it into the repository's expected dataset path.

In [ ]:
from google.colab import drive

if TERMS_ACKNOWLEDGMENT != 'I ACCEPT AGRICULTURE-VISION TERMS':
    raise RuntimeError('Terms acknowledgment is required in this session.')
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/shepherd-ai-private/week6')
DATASET_DIR = DRIVE_ROOT / 'agriculture-vision'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
DATASET_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
REPO_DATASET_DIR = REPO / 'datasets' / 'aerial_images' / 'agriculture-vision'
if REPO_DATASET_DIR.is_symlink():
    REPO_DATASET_DIR.unlink()
elif REPO_DATASET_DIR.exists():
    if any(REPO_DATASET_DIR.iterdir()):
        raise RuntimeError(f'Refusing to replace non-empty runtime dataset directory: {REPO_DATASET_DIR}')
    REPO_DATASET_DIR.rmdir()
REPO_DATASET_DIR.symlink_to(DATASET_DIR, target_is_directory=True)
print('private_drive_cache', DRIVE_ROOT)
print('repository_dataset_link', REPO_DATASET_DIR, '->', REPO_DATASET_DIR.resolve())


## 5. Cache And Extract The Official 2017 Archive

The archive is approximately 1.88 GB. It is downloaded from the official IntelinAir AWS bucket only when absent from Drive. Its SHA-256 is recalculated every session and written to a non-image provenance record that may be committed to GitHub.

In [ ]:
import hashlib
import json
import tarfile
from urllib.request import urlretrieve

BASE = 'https://intelinair-data-releases.s3.amazonaws.com/agriculture-vision/cvpr_paper_2020/Dataset'
ARCHIVE = DATASET_DIR / 'data2017_miniscale.tar.gz'
SPLITS = DATASET_DIR / 'data2017_splits.json'
if not ARCHIVE.exists():
    urlretrieve(f'{BASE}/data2017_miniscale.tar.gz', ARCHIVE)
if not SPLITS.exists():
    urlretrieve(f'{BASE}/data2017_splits.json', SPLITS)
digest = hashlib.sha256()
with ARCHIVE.open('rb') as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b''):
        digest.update(chunk)
ARCHIVE_SHA256 = digest.hexdigest()
print('archive_sha256', ARCHIVE_SHA256)
EXTRACTED = DATASET_DIR / 'data2017'
EXTRACTED.mkdir(exist_ok=True)
EXPECTED_RGB_TILES = 8345
rgb_tiles = list(EXTRACTED.rglob('field_images/rgb/*.jpg'))
if len(rgb_tiles) != EXPECTED_RGB_TILES:
    print(f'Repairing incomplete extraction: found {len(rgb_tiles)} of {EXPECTED_RGB_TILES} RGB tiles')
    with tarfile.open(ARCHIVE, 'r:gz') as archive:
        archive.extractall(EXTRACTED, filter='data')
    rgb_tiles = list(EXTRACTED.rglob('field_images/rgb/*.jpg'))
if len(rgb_tiles) != EXPECTED_RGB_TILES:
    raise RuntimeError(f'Extraction validation failed: found {len(rgb_tiles)} of {EXPECTED_RGB_TILES} RGB tiles')
print('rgb_tiles', len(rgb_tiles))
print('extracted_to', EXTRACTED)
PROVENANCE = REPO / 'outputs' / 'evaluations' / 'week6_agriculture_vision_cache_provenance.json'
PROVENANCE.parent.mkdir(parents=True, exist_ok=True)
PROVENANCE.write_text(json.dumps({
    'source_url': f'{BASE}/data2017_miniscale.tar.gz',
    'archive_filename': ARCHIVE.name,
    'archive_sha256': ARCHIVE_SHA256,
    'split_filename': SPLITS.name,
    'storage': 'private_google_drive_cache',
    'licensed_pixels_committed': False,
}, indent=2) + '\n', encoding='utf-8')
print('provenance_record', PROVENANCE)


## 6. Record The Extracted Dataset Layout

Record relative paths and counts before implementing a label loader. This is the evidence for the 2017 class-mask and valid-region mapping; it does not copy licensed pixels or infer label semantics.

In [ ]:
!python scripts/inspect_agriculture_vision_layout.py --dataset-dir datasets/aerial_images/agriculture-vision/data2017 --output outputs/evaluations/week6_agriculture_vision_layout.json --sample-limit 120


## 7. Build And Validate A Leakage-Safe Subset Manifest

The official farmland-level split JSON is authoritative. Selection is deterministic and limited to 10 RGB images per split. Every selected image receives a SHA-256 digest.

In [ ]:
subprocess.run(['python', 'scripts/prepare_agriculture_vision_subset.py', '--dataset-dir', str(EXTRACTED), '--dataset-root', str(DRIVE_ROOT), '--split-json', str(SPLITS), '--output', 'datasets/aerial_images/manifest.jsonl', '--max-per-split', '10', '--accept-terms'], check=True)
subprocess.run(['python', 'scripts/validate_vision_manifest.py', '--manifest', 'datasets/aerial_images/manifest.jsonl', '--dataset-root', str(DRIVE_ROOT), '--summary-output', 'outputs/evaluations/week6_vision_manifest_summary.json'], check=True)


## 8. Audit Train And Validation Labels

Validate aligned masks and record class prevalence and overlap before training. This command rejects the test split so final evaluation labels remain untouched.

In [ ]:
subprocess.run(['python', 'scripts/validate_agriculture_vision_labels.py', '--manifest', 'datasets/aerial_images/manifest.jsonl', '--dataset-root', str(DRIVE_ROOT), '--labels-dir', str(EXTRACTED / 'data2017_miniscale'), '--split', 'train', '--split', 'validation', '--output', 'outputs/evaluations/week6_agriculture_vision_label_audit.json'], check=True)


## 9. Export Non-Image CPU Audit Artifacts

Package the manifest, provenance, layout, and label-audit records with checksums. The bundle contains no licensed pixels or model weights and is copied to the private Drive run directory for persistence.

In [ ]:
CPU_ARTIFACT_ZIP = DRIVE_ROOT / 'week6_cpu_audit_artifacts.zip'
subprocess.run(['python', 'scripts/package_week6_cpu_artifacts.py', '--repo-root', str(REPO), '--output-zip', str(CPU_ARTIFACT_ZIP)], check=True)
print('cpu_artifact_zip', CPU_ARTIFACT_ZIP)


## 10. Run The YOLO Smoke-Test Baseline

Detection counts and confidence values are pipeline outputs, not Agriculture-Vision anomaly performance. Zero detections are preserved as a valid negative result.

In [ ]:
if not HAS_T4:
    raise RuntimeError('YOLO inference requires a T4. CPU debug steps 1-8 are still valid; resume this cell when T4 quota is available.')
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[vision]'], check=True)
subprocess.run(['python', 'scripts/run_yolo_detection.py', '--manifest', 'datasets/aerial_images/manifest.jsonl', '--dataset-root', str(DRIVE_ROOT), '--model', 'yolov8n.pt', '--confidence', '0.25', '--device', '0', '--required-device-substring', 'T4', '--predictions-output', 'outputs/evaluations/week6_yolo_detections.jsonl', '--summary-output', 'outputs/evaluations/week6_yolo_detection_summary.json', '--annotated-dir', 'outputs/visualizations/week6_yolo'], check=True)


## 11. Train The Segmentation Development Baseline

Rebuild a 64-record-per-split development manifest, train only on `train`, select checkpoints only on `validation`, and never load test masks. This three-epoch U-Net run validates the research pipeline; it is not final benchmark performance. Checkpoints and metrics persist in private Drive storage.

In [ ]:
if not HAS_T4:
    raise RuntimeError('Segmentation training requires a T4.')
subprocess.run(['python', 'scripts/prepare_agriculture_vision_subset.py', '--dataset-dir', str(EXTRACTED), '--dataset-root', str(DRIVE_ROOT), '--split-json', str(SPLITS), '--output', 'datasets/aerial_images/manifest_segmentation_dev64.jsonl', '--max-per-split', '64', '--accept-terms'], check=True)
SEGMENTATION_OUTPUT = CHECKPOINT_DIR / 'agriculture_vision_small_unet_dev64'
subprocess.run(['python', 'scripts/train_agriculture_vision_segmentation.py', '--manifest', 'datasets/aerial_images/manifest_segmentation_dev64.jsonl', '--dataset-root', str(DRIVE_ROOT), '--labels-dir', str(EXTRACTED / 'data2017_miniscale'), '--output-dir', str(SEGMENTATION_OUTPUT), '--epochs', '3', '--batch-size', '4', '--learning-rate', '0.001', '--base-channels', '16', '--seed', '17', '--num-workers', '2', '--required-device-substring', 'T4', '--resume'], check=True)
print('segmentation_output', SEGMENTATION_OUTPUT)


## 12. Audit A Seeded Follow-Up Development Sample

The sorted-prefix baseline was dominated by background. Build a larger deterministic hash-ranked sample and audit its train/validation class coverage before changing the loss or starting another training run. This step still does not load test masks.

In [ ]:
subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
SEEDED_MANIFEST = REPO / 'datasets/aerial_images/manifest_segmentation_seed17_dev256.jsonl'
subprocess.run(['python', 'scripts/prepare_agriculture_vision_subset.py', '--dataset-dir', str(EXTRACTED), '--dataset-root', str(DRIVE_ROOT), '--split-json', str(SPLITS), '--output', str(SEEDED_MANIFEST), '--max-per-split', '256', '--selection-strategy', 'seeded-hash', '--selection-seed', '17', '--accept-terms'], cwd=REPO, check=True)
SEEDED_AUDIT = REPO / 'outputs/evaluations/week6_agriculture_vision_seed17_dev256_label_audit.json'
subprocess.run(['python', 'scripts/validate_agriculture_vision_labels.py', '--manifest', str(SEEDED_MANIFEST), '--dataset-root', str(DRIVE_ROOT), '--labels-dir', str(EXTRACTED / 'data2017_miniscale'), '--split', 'train', '--split', 'validation', '--output', str(SEEDED_AUDIT)], cwd=REPO, check=True)
print(SEEDED_AUDIT.read_text())


## 13. Audit The Complete Development Labels

The 256/256 sample can omit rare classes. Build a manifest containing every official split record, audit all train/validation labels, and create a separate train-only audit for loss weighting. Test labels remain excluded.

In [ ]:
FULL_MANIFEST = REPO / 'datasets/aerial_images/manifest_segmentation_seed17_full.jsonl'
subprocess.run(['python', 'scripts/prepare_agriculture_vision_subset.py', '--dataset-dir', str(EXTRACTED), '--dataset-root', str(DRIVE_ROOT), '--split-json', str(SPLITS), '--output', str(FULL_MANIFEST), '--max-per-split', '10000', '--selection-strategy', 'seeded-hash', '--selection-seed', '17', '--accept-terms'], cwd=REPO, check=True)
FULL_AUDIT = REPO / 'outputs/evaluations/week6_agriculture_vision_full_train_validation_label_audit.json'
subprocess.run(['python', 'scripts/validate_agriculture_vision_labels.py', '--manifest', str(FULL_MANIFEST), '--dataset-root', str(DRIVE_ROOT), '--labels-dir', str(EXTRACTED / 'data2017_miniscale'), '--split', 'train', '--split', 'validation', '--output', str(FULL_AUDIT)], cwd=REPO, check=True)
TRAIN_AUDIT = REPO / 'outputs/evaluations/week6_agriculture_vision_full_train_label_audit.json'
subprocess.run(['python', 'scripts/validate_agriculture_vision_labels.py', '--manifest', str(FULL_MANIFEST), '--dataset-root', str(DRIVE_ROOT), '--labels-dir', str(EXTRACTED / 'data2017_miniscale'), '--split', 'train', '--output', str(TRAIN_AUDIT)], cwd=REPO, check=True)
print(TRAIN_AUDIT.read_text())


## 14. Run The Imbalance-Aware T4 Comparison

Use the train-only audit to derive capped positive weights and exclude channels with zero training positives. This keeps validation independent. Run on T4 and compare modified mIoU against the unweighted baseline; do not compare weighted objective loss directly with unweighted baseline loss.

In [ ]:
TRAIN_AUDIT = REPO / 'outputs/evaluations/week6_agriculture_vision_full_train_label_audit.json'
if not TRAIN_AUDIT.exists():
    raise FileNotFoundError(f'Run Section 13 or pull the tracked train-only audit: {TRAIN_AUDIT}')
if not HAS_T4:
    raise RuntimeError('The imbalance-aware research comparison requires a T4. CPU timing runs must use a separate output directory and --device cpu.')
WEIGHTED_OUTPUT = CHECKPOINT_DIR / 'agriculture_vision_small_unet_dev64_weightcap20'
subprocess.run(['python', 'scripts/train_agriculture_vision_segmentation.py', '--manifest', 'datasets/aerial_images/manifest_segmentation_dev64.jsonl', '--dataset-root', str(DRIVE_ROOT), '--labels-dir', str(EXTRACTED / 'data2017_miniscale'), '--output-dir', str(WEIGHTED_OUTPUT), '--epochs', '3', '--batch-size', '4', '--learning-rate', '0.001', '--base-channels', '16', '--seed', '17', '--num-workers', '2', '--device', 'cuda', '--required-device-substring', 'T4', '--train-label-audit', str(TRAIN_AUDIT), '--positive-weight-cap', '20', '--resume'], cwd=REPO, check=True)
print((WEIGHTED_OUTPUT / 'metrics.json').read_text())


## 15. Run The Fixed Seed-17 256/256 T4 Baseline

Scale the unweighted development baseline to the preregistered hash-ranked 256-train/256-validation subset. Training reads only official training records; evaluation reads only official validation records; test labels remain unused. The primary metric is validation modified mIoU. Compare this run with the 64/64 unweighted T4 baseline, but preserve the result whether it improves or regresses.

In [ ]:
if not HAS_T4:
    raise RuntimeError('The fixed 256/256 comparison requires a T4 runtime.')
SEEDED_MANIFEST = REPO / 'datasets/aerial_images/manifest_segmentation_seed17_dev256.jsonl'
if not SEEDED_MANIFEST.exists():
    raise FileNotFoundError(f'Run Section 12 to create the fixed subset: {SEEDED_MANIFEST}')
SCALED_OUTPUT = CHECKPOINT_DIR / 'agriculture_vision_small_unet_seed17_dev256_unweighted'
subprocess.run(['python', 'scripts/train_agriculture_vision_segmentation.py', '--manifest', str(SEEDED_MANIFEST), '--dataset-root', str(DRIVE_ROOT), '--labels-dir', str(EXTRACTED / 'data2017_miniscale'), '--output-dir', str(SCALED_OUTPUT), '--epochs', '5', '--batch-size', '4', '--learning-rate', '0.001', '--base-channels', '16', '--seed', '17', '--num-workers', '2', '--device', 'cuda', '--required-device-substring', 'T4', '--resume'], cwd=REPO, check=True)
print((SCALED_OUTPUT / 'metrics.json').read_text())


## 16. Compare BCE With Anomaly-Only Soft Dice

The fixed 256/256 BCE run converged but predicted only background at its best epoch. Keep the model, subset, seed, optimizer, and evaluation unchanged. Add soft Dice over anomaly channels that have positives in the complete training split; exclude background and train-absent channels from Dice. Validation labels do not influence the objective.

In [ ]:
subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
if not HAS_T4:
    raise RuntimeError('The BCE-Dice comparison requires a T4 runtime.')
SEEDED_MANIFEST = REPO / 'datasets/aerial_images/manifest_segmentation_seed17_dev256.jsonl'
TRAIN_AUDIT = REPO / 'outputs/evaluations/week6_agriculture_vision_full_train_label_audit.json'
for required_path in (SEEDED_MANIFEST, TRAIN_AUDIT):
    if not required_path.exists():
        raise FileNotFoundError(required_path)
DICE_OUTPUT = CHECKPOINT_DIR / 'agriculture_vision_small_unet_seed17_dev256_bce_dice1'
subprocess.run(['python', 'scripts/train_agriculture_vision_segmentation.py', '--manifest', str(SEEDED_MANIFEST), '--dataset-root', str(DRIVE_ROOT), '--labels-dir', str(EXTRACTED / 'data2017_miniscale'), '--output-dir', str(DICE_OUTPUT), '--epochs', '5', '--batch-size', '4', '--learning-rate', '0.001', '--base-channels', '16', '--seed', '17', '--num-workers', '2', '--device', 'cuda', '--required-device-substring', 'T4', '--train-label-audit', str(TRAIN_AUDIT), '--loss', 'bce-dice', '--dice-weight', '1.0', '--resume'], cwd=REPO, check=True)
print((DICE_OUTPUT / 'metrics.json').read_text())


## 17. Compare A Train-Label-Stratified 256-Image Subset

The fixed training subset contained no endrow, storm-damage, or water pixels. Build a new 256-record training subset using training labels only: reserve up to four positive records for each available anomaly class, then fill by seed-17 hash rank. Keep the validation selection fixed and keep the BCE-Dice model configuration unchanged.

In [ ]:
subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
STRATIFIED_MANIFEST = REPO / 'datasets/aerial_images/manifest_segmentation_seed17_train_stratified_dev256.jsonl'
LABEL_PRESENCE_CACHE = REPO / 'outputs/evaluations/week6_agriculture_vision_train_label_presence.json'
subprocess.run(['python', 'scripts/prepare_agriculture_vision_subset.py', '--dataset-dir', str(EXTRACTED), '--dataset-root', str(DRIVE_ROOT), '--split-json', str(SPLITS), '--output', str(STRATIFIED_MANIFEST), '--max-per-split', '256', '--selection-strategy', 'train-label-stratified', '--selection-seed', '17', '--labels-dir', str(EXTRACTED / 'data2017_miniscale'), '--min-positive-records-per-class', '4', '--label-presence-cache', str(LABEL_PRESENCE_CACHE), '--label-scan-workers', '16', '--accept-terms'], cwd=REPO, check=True)
STRATIFIED_TRAIN_AUDIT = REPO / 'outputs/evaluations/week6_seed17_train_stratified_dev256_train_label_audit.json'
subprocess.run(['python', 'scripts/validate_agriculture_vision_labels.py', '--manifest', str(STRATIFIED_MANIFEST), '--dataset-root', str(DRIVE_ROOT), '--labels-dir', str(EXTRACTED / 'data2017_miniscale'), '--split', 'train', '--output', str(STRATIFIED_TRAIN_AUDIT)], cwd=REPO, check=True)
if not HAS_T4:
    raise RuntimeError('The stratified BCE-Dice comparison requires a T4 runtime.')
STRATIFIED_OUTPUT = CHECKPOINT_DIR / 'agriculture_vision_small_unet_seed17_train_stratified_dev256_bce_dice1'
subprocess.run(['python', 'scripts/train_agriculture_vision_segmentation.py', '--manifest', str(STRATIFIED_MANIFEST), '--dataset-root', str(DRIVE_ROOT), '--labels-dir', str(EXTRACTED / 'data2017_miniscale'), '--output-dir', str(STRATIFIED_OUTPUT), '--epochs', '5', '--batch-size', '4', '--learning-rate', '0.001', '--base-channels', '16', '--seed', '17', '--num-workers', '2', '--device', 'cuda', '--required-device-substring', 'T4', '--train-label-audit', str(STRATIFIED_TRAIN_AUDIT), '--loss', 'bce-dice', '--dice-weight', '1.0', '--resume'], cwd=REPO, check=True)
print((STRATIFIED_OUTPUT / 'metrics.json').read_text())


## 18. Resume The Registered VisDrone Detector

This is the labeled YOLO-compatible detection experiment required alongside Agriculture-Vision segmentation. Run it in a fresh T4 runtime so the exact registered Ultralytics version is imported. It uses the official VisDrone train and validation splits, refuses CPU training, verifies the package version, device, saved arguments, and checkpoint epoch before resuming, checkpoints every epoch to private Drive, and does not use test-dev for model selection. The registered configuration is documented in `docs/week6_visdrone_detection_protocol.md`.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO = Path('/content/shepherd-ai')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', 'codex/week6-vision', 'https://github.com/cyberuniversal/shepherd-ai.git', str(REPO)], check=True)
else:
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
os.chdir(REPO)

# Install before importing Ultralytics; rerun only in a fresh runtime.
subprocess.run(['python', '-m', 'pip', 'install', '-q', 'ultralytics==8.4.92'], check=True)

import torch
import yaml
import ultralytics
from google.colab import drive
from ultralytics import YOLO

if ultralytics.__version__ != '8.4.92':
    raise RuntimeError(f'Expected Ultralytics 8.4.92, found {ultralytics.__version__}. Restart the runtime before retrying.')
if not torch.cuda.is_available() or 'T4' not in torch.cuda.get_device_name(0):
    raise RuntimeError('The registered VisDrone baseline requires an NVIDIA T4. Do not train it on CPU.')

drive.mount('/content/drive', force_remount=False)
VISDRONE_RUN = Path('/content/drive/MyDrive/shepherd-ai-private/week6/checkpoints/visdrone_yolov8n_seed17_e50')
VISDRONE_LAST = VISDRONE_RUN / 'weights' / 'last.pt'
VISDRONE_ARGS = VISDRONE_RUN / 'args.yaml'
for required_path in (VISDRONE_LAST, VISDRONE_ARGS):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

print('ultralytics_version', ultralytics.__version__)
print('torch_version', torch.__version__)
print('device', torch.cuda.get_device_name(0))
saved_args = yaml.safe_load(VISDRONE_ARGS.read_text(encoding='utf-8'))
expected_args = {'epochs': 50, 'imgsz': 640, 'batch': 16, 'seed': 17, 'deterministic': True}
mismatches = {key: {'expected': value, 'observed': saved_args.get(key)} for key, value in expected_args.items() if saved_args.get(key) != value}
if mismatches:
    raise RuntimeError(f'Refusing to resume a mismatched VisDrone run: {mismatches}')
checkpoint = torch.load(VISDRONE_LAST, map_location='cpu', weights_only=False)
completed_epochs = int(checkpoint['epoch']) + 1
if not 20 <= completed_epochs <= 50:
    raise RuntimeError(f'Expected a checkpoint from completed epoch 20 through 50, found {completed_epochs}')
print('resuming_after_completed_epoch', completed_epochs)
if completed_epochs < 50:
    YOLO(str(VISDRONE_LAST)).train(resume=True)
else:
    print('The registered 50-epoch training budget is already complete; no resume was started.')
